# A-001 Simple SQL Agent + RAG
## No LlamaIndex. No Pydantic. No Hugging Face.

This version is intentionally simple and stable for Databricks training.

### Flow

`Question`
→ `GPT-OSS 120B creates SQL`
→ `Spark SQL gets structured facts`
→ `Qwen3 embeds the question`
→ `NumPy retrieves top PDF chunks`
→ `GPT-OSS 120B combines SQL + PDF evidence`

### Models

- Embedding: `databricks-qwen3-embedding-0-6b`
- Reasoning/chat: `databricks-gpt-oss-120b`

### Why this version

We remove:
- LlamaIndex
- Pydantic
- Hugging Face
- sentence-transformers

So the notebook has fewer moving parts and avoids the Pydantic schema error.

In [0]:
# CELL 1 — Install only simple libraries

%pip install -q pypdf numpy mlflow

# If this is the first installation in the notebook session:
# dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## Step 1 — Configuration

Change these two values only if your Databricks folder/table names are different.

In [0]:
# CELL 2 — Configuration

from pathlib import Path
import json
import re
import numpy as np
from pypdf import PdfReader
from mlflow.deployments import get_deploy_client


DOCUMENT_FOLDER = "/Workspace/Nuclear_Enterprise_360/A001 Documents"
ASSET_TABLE = "workspace.nuclear_enterprise_360.asset_360"

EMBED_MODEL = "databricks-qwen3-embedding-0-6b"
CHAT_MODEL = "databricks-gpt-oss-120b"

CHUNK_SIZE = 400
CHUNK_OVERLAP = 100
TOP_K = 3

client = get_deploy_client("databricks")

print("Ready")
print("PDF folder :", DOCUMENT_FOLDER)
print("SQL table  :", ASSET_TABLE)
print("Embedding  :", EMBED_MODEL)
print("LLM        :", CHAT_MODEL)

Ready
PDF folder : /Workspace/Nuclear_Enterprise_360/A001 Documents
SQL table  : workspace.nuclear_enterprise_360.asset_360
Embedding  : databricks-qwen3-embedding-0-6b
LLM        : databricks-gpt-oss-120b


## Step 2 — Small helper for Databricks embeddings

We call the Databricks Qwen endpoint directly.

No Pydantic model.
No custom LlamaIndex class.

In [0]:
# CELL 3 — Databricks Qwen embedding

def embed_texts(texts):
    """
    texts: list[str]
    returns: list[list[float]]
    """
    response = client.predict(
        endpoint=EMBED_MODEL,
        inputs={"input": texts}
    )

    data = response["data"] if isinstance(response, dict) else response.data

    vectors = []
    for item in data:
        if isinstance(item, dict):
            vectors.append(item["embedding"])
        else:
            vectors.append(item.embedding)

    return vectors


# Small health test
test_vector = embed_texts(["A-001 inspection"])[0]

print("Embedding works")
print("Dimension:", len(test_vector))

Embedding works
Dimension: 1024


## Step 3 — Read PDFs and create simple chunks

This is plain Python.

For each PDF:
1. extract text
2. split into overlapping chunks
3. remember source filename

In [0]:
# CELL 4 — Read PDFs + chunk them

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    text = " ".join(text.split())

    if not text:
        return []

    chunks = []
    start = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])

        if end == len(text):
            break

        start = end - overlap

    return chunks


def load_pdf_chunks(folder):
    records = []

    pdf_files = list(Path(folder).glob("*.pdf"))

    print("PDF files found:", len(pdf_files))

    for pdf_file in pdf_files:
        reader = PdfReader(str(pdf_file))

        for page_number, page in enumerate(reader.pages, start=1):
            text = page.extract_text() or ""

            for chunk_number, chunk in enumerate(chunk_text(text), start=1):
                records.append({
                    "source": pdf_file.name,
                    "page": page_number,
                    "chunk": chunk_number,
                    "text": chunk
                })

    return records


chunks = load_pdf_chunks(DOCUMENT_FOLDER)

print("Total chunks:", len(chunks))
print("Example:")
print(chunks[0] if chunks else "No chunks found")

PDF files found: 9
Total chunks: 100
Example:
{'source': '08_WO-2026-0817_A001_Inspection_Work_Order.pdf', 'page': 1, 'chunk': 1, 'text': 'NORTHSTAR PROCESS FACILITY SYNTHETIC TRAINING CORPUS WO-2026-0817 | OPEN | Synthetic training document - not for operational use Page 1 WO-2026-0817 / OPEN Inspection Work Order - A-001 Elevated Vibration Review A-001 | Cooling Water Pump | Enterprise Asset Reliability Training Scenario Document ID WO-2026-0817 Version 1.0 Status OPEN Effective / Issue Date 2026-08-21 Owner Maintenance Planning As'}


## Step 4 — Embed the document chunks once

For a small classroom demo, we keep embeddings in memory.

Later, if required, these vectors can be saved to a file or Unity Catalog Volume.

In [0]:
# CELL 5 — Create document embeddings

BATCH_SIZE = 8

all_vectors = []

for start in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[start:start + BATCH_SIZE]
    batch_texts = [x["text"] for x in batch]

    vectors = embed_texts(batch_texts)
    all_vectors.extend(vectors)

document_matrix = np.array(all_vectors, dtype=np.float32)

# Normalize once for cosine similarity
norms = np.linalg.norm(document_matrix, axis=1, keepdims=True)
norms[norms == 0] = 1
document_matrix = document_matrix / norms

print("Document embeddings ready")
print("Shape:", document_matrix.shape)

Document embeddings ready
Shape: (100, 1024)


## Step 5 — Simple RAG retriever

The question is embedded with the **same Qwen model**.

Then NumPy calculates cosine similarity and returns the top 3 chunks.

In [0]:
# CELL 6 — Simple vector retrieval

def retrieve(question, top_k=TOP_K, verbose=True):
    q = np.array(embed_texts([question])[0], dtype=np.float32)

    q_norm = np.linalg.norm(q)

    if q_norm == 0:
        raise ValueError("Question embedding has zero norm.")

    q = q / q_norm

    scores = document_matrix @ q

    top_k = min(top_k, len(chunks))
    indices = np.argsort(scores)[::-1][:top_k]

    results = []

    if verbose:
        print("\n" + "=" * 65)
        print("RAG RETRIEVER")
        print("=" * 65)
        print("Embedding model:", EMBED_MODEL)
        print("Top K:", top_k)

    for rank, idx in enumerate(indices, start=1):
        item = dict(chunks[int(idx)])
        item["score"] = float(scores[int(idx)])
        results.append(item)

        if verbose:
            print(f"\n[{rank}] {item['source']} | page {item['page']}")
            print("Similarity:", round(item["score"], 4))
            print("Preview:", item["text"][:300], "...")

    return results

## Step 6 — GPT-OSS 120B helper

We use GPT-OSS 120B to:
- create SQL
- combine SQL + RAG evidence

`reasoning_effort="medium"` asks the model to use more reasoning internally.

The notebook prints the **tool trace** but not hidden chain-of-thought.

In [0]:
# CELL 7 — GPT-OSS 120B

def call_llm(messages, max_tokens=900):
    response = client.predict(
        endpoint=CHAT_MODEL,
        inputs={
            "messages": messages,
            "temperature": 0.1,
            "max_tokens": max_tokens,
            "reasoning_effort": "medium"
        }
    )

    if isinstance(response, dict):
        return response["choices"][0]["message"]["content"]

    return response.choices[0].message.content

## Step 7 — Simple SQL Agent

The agent receives the **actual Databricks table schema**, generates one SELECT query, validates it, and runs it using Spark SQL.

This is intentionally read-only for training.

In [0]:
# CELL 8 — SQL Agent

def get_schema():
    return "\n".join(
        f"- {name}: {dtype}"
        for name, dtype in spark.table(ASSET_TABLE).dtypes
    )


def clean_sql(sql):
    sql = sql.strip()
    sql = re.sub(r"^```(?:sql)?", "", sql, flags=re.I).strip()
    sql = re.sub(r"```$", "", sql).strip()
    return sql.rstrip(";").strip()


def validate_sql(sql):
    upper = f" {sql.upper()} "

    if not sql.upper().startswith("SELECT"):
        raise ValueError("Only SELECT is allowed.")

    forbidden = [
        " INSERT ",
        " UPDATE ",
        " DELETE ",
        " DROP ",
        " ALTER ",
        " CREATE ",
        " MERGE ",
        " TRUNCATE "
    ]

    for word in forbidden:
        if word in upper:
            raise ValueError("Write operations are not allowed.")

    if ASSET_TABLE.lower() not in sql.lower():
        raise ValueError(
            f"Only this table may be queried: {ASSET_TABLE}"
        )


def sql_agent(question, verbose=True):

    if verbose:
        print("\n" + "=" * 65)
        print("SQL AGENT")
        print("=" * 65)
        print("Model:", CHAT_MODEL)
        print("Table:", ASSET_TABLE)

    prompt = f"""
You are a Databricks Spark SQL agent.

TABLE:
{ASSET_TABLE}

SCHEMA:
{get_schema()}

USER QUESTION:
{question}

Rules:
- Return ONE SELECT statement only.
- Use only {ASSET_TABLE}.
- Never modify data.
- Prefer LIMIT 20.
- Asset IDs may look like A-001.
- Return SQL only.
- No markdown.
- No explanation.
"""

    sql = call_llm(
        [
            {
                "role": "system",
                "content": "Generate safe Spark SQL only."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        max_tokens=400
    )

    sql = clean_sql(sql)
    validate_sql(sql)

    if verbose:
        print("\nGENERATED SQL")
        print(sql)

    rows = spark.sql(sql).limit(20).collect()
    rows = [row.asDict(recursive=True) for row in rows]

    if verbose:
        print("\nSQL RESULT")
        print(json.dumps(rows, indent=2, default=str))

    return {
        "sql": sql,
        "rows": rows
    }

## Step 8 — Hybrid SQL + RAG Agent

This is the only function learners need to call:

```python
ask("question", verbose=True)
```

The visible workflow is:

**question → SQL → database → RAG retrieval → GPT-OSS final answer**

In [0]:
# CELL 9 — Main hybrid agent

def ask(question, verbose=True):

    if verbose:
        print("\n" + "#" * 65)
        print("A001 SQL + RAG AGENT")
        print("#" * 65)
        print("Question:", question)
        print("Reasoning model:", CHAT_MODEL)
        print("Embedding model:", EMBED_MODEL)
        print("Plan: SQL + RAG -> grounded answer")

    # Structured data
    sql_result = sql_agent(
        question,
        verbose=verbose
    )

    # Documents
    rag_results = retrieve(
        question,
        top_k=TOP_K,
        verbose=verbose
    )

    rag_text = "\n\n".join(
        f"SOURCE: {x['source']} | PAGE: {x['page']}\n"
        f"SIMILARITY: {x['score']}\n"
        f"TEXT: {x['text']}"
        for x in rag_results
    )

    if verbose:
        print("\n" + "=" * 65)
        print("FINAL SYNTHESIS")
        print("=" * 65)
        print("Model:", CHAT_MODEL)

    final_prompt = f"""
QUESTION:
{question}

STRUCTURED SQL EVIDENCE

SQL:
{sql_result['sql']}

ROWS:
{json.dumps(sql_result['rows'], indent=2, default=str)}

DOCUMENT EVIDENCE:
{rag_text}

Rules:
1. Use only the evidence supplied above.
2. SQL is authoritative for current structured asset values.
3. For procedures, prefer APPROVED documents.
4. DRAFT or SUPERSEDED documents are not current authority.
5. If evidence is missing, say so.
6. Mention relevant document names/pages when available.
7. Do not invent maintenance actions or diagnoses.
8. Finish with a short human next step.
"""

    answer = call_llm(
        [
            {
                "role": "system",
                "content": (
                    "You are an enterprise A001 assistant. "
                    "Use only supplied SQL and document evidence."
                )
            },
            {
                "role": "user",
                "content": final_prompt
            }
        ],
        max_tokens=1000
    )

    print("\nFINAL ANSWER\n")
    print(answer)

    return answer

# Step 9 — Test the database first

This proves `A-001` exists in your structured data.

In [0]:
# CELL 10 — Database sanity check

display(
    spark.sql(
        f"""
        SELECT *
        FROM {ASSET_TABLE}
        WHERE asset_id = 'A-001'
        LIMIT 5
        """
    )
)

asset_id,asset_name,asset_type,criticality,status,system_name,unit_id,health_score,risk_level,recommended_action,open_work_orders,high_priority_open_work,follow_up_findings,latest_inspection_date
A-001,Pump 001,Pump,HIGH,IN_SERVICE,Cooling Water,TRN-A,68.05,MEDIUM,Increase monitoring,6,3,2,2026-08-23


# Step 10 — Test SQL + RAG together

This question deliberately requires:
- SQL facts from the table
- inspection/procedure evidence from the PDFs

In [0]:
# CELL 11 — Hybrid test (FIXED)

# GPT-OSS may return content as a list instead of a normal string.
# This helper converts it safely into text before the SQL Agent uses .strip().

def call_llm(messages, max_tokens=900):

    response = client.predict(
        endpoint=CHAT_MODEL,
        inputs={
            "messages": messages,
            "temperature": 0.1,
            "max_tokens": max_tokens,
            "reasoning_effort": "medium"
        }
    )

    # Get message content
    if isinstance(response, dict):
        content = response["choices"][0]["message"]["content"]
    else:
        content = response.choices[0].message.content

    # Normal string response
    if isinstance(content, str):
        return content

    # GPT-OSS can return a list of content blocks
    if isinstance(content, list):

        text_parts = []

        for block in content:

            if isinstance(block, dict):
                text = block.get("text")

                if text:
                    text_parts.append(str(text))

            else:
                text = getattr(block, "text", None)

                if text:
                    text_parts.append(str(text))

        return "\n".join(text_parts).strip()

    # Final fallback
    return str(content)


# ------------------------------------------------------------
# TEST QUESTION
# ------------------------------------------------------------

question = """
For A-001, what are the current risk level, health score and open work orders
from the asset database, and what approved inspection or reliability procedure
applies according to the documents?
"""


# ------------------------------------------------------------
# RUN THE SQL + RAG AGENT
# ------------------------------------------------------------

answer = ask(
    question,
    verbose=True
)

print("\n" + "=" * 70)
print("FINAL RESULT")
print("=" * 70)
print(answer)


#################################################################
A001 SQL + RAG AGENT
#################################################################
Question: 
For A-001, what are the current risk level, health score and open work orders
from the asset database, and what approved inspection or reliability procedure
applies according to the documents?

Reasoning model: databricks-gpt-oss-120b
Embedding model: databricks-qwen3-embedding-0-6b
Plan: SQL + RAG -> grounded answer

SQL AGENT
Model: databricks-gpt-oss-120b
Table: workspace.nuclear_enterprise_360.asset_360

GENERATED SQL
SELECT asset_id,
       risk_level,
       health_score,
       open_work_orders,
       recommended_action AS approved_inspection_procedure
FROM workspace.nuclear_enterprise_360.asset_360
WHERE asset_id = 'A-001'
LIMIT 20

SQL RESULT
[
  {
    "asset_id": "A-001",
    "risk_level": "MEDIUM",
    "health_score": 68.05,
    "open_work_orders": 6,
    "approved_inspection_procedure": "Increase monitoring"
  

# Optional learner question box

In [0]:
# CELL 12 — Interactive notebook input

dbutils.widgets.text(
    "A001_question",
    "What is the status of A-001 and what do the approved documents say about its inspection?"
)

user_question = dbutils.widgets.get("A001_question")

ask(
    user_question,
    verbose=True
)


#################################################################
A001 SQL + RAG AGENT
#################################################################
Question: What is the status of A-001 and what do the approved documents say about its inspection?
Reasoning model: databricks-gpt-oss-120b
Embedding model: databricks-qwen3-embedding-0-6b
Plan: SQL + RAG -> grounded answer

SQL AGENT
Model: databricks-gpt-oss-120b
Table: workspace.nuclear_enterprise_360.asset_360

GENERATED SQL
SELECT asset_id, status, latest_inspection_date
FROM workspace.nuclear_enterprise_360.asset_360
WHERE asset_id = 'A-001'
LIMIT 20

SQL RESULT
[
  {
    "asset_id": "A-001",
    "status": "IN_SERVICE",
    "latest_inspection_date": "2026-08-23"
  }
]

RAG RETRIEVER
Embedding model: databricks-qwen3-embedding-0-6b
Top K: 3

[1] 04_A001-PROC-INS-001-V2_Approved_Inspection_and_PM_Procedure.pdf | page 2
Similarity: 0.6175
Preview: or quality, inspect recent work/finding history and request qualified maintenance/rel

'**Asset status (SQL evidence)**  \n- **Asset ID:** A‑001  \n- **Current status:** **IN_SERVICE**  \n- **Latest inspection date:** 2026‑08‑23  \n\n**What the approved inspection procedure says (approved document evidence)**  \n\n- The approved procedure **“04_A001‑PROC‑INS‑001‑V2_Approved_Inspection_and_PM_Procedure.pdf”** (page\u202f2) defines the inspection workflow for A‑001. It requires the inspector to:  \n  1. **Verify sensor quality** before relying on vibration data.  \n  2. **Inspect recent work/finding history** to understand any open work orders or recent maintenance actions.  \n  3. **Request a qualified maintenance/reliability review** when the data shows a concerning pattern (e.g., lower health score, upward vibration trend, open work orders).  \n\n- The same page notes that, for A‑001, the observed condition (lower health score, sustained upward vibration trend, several open work orders) “**warrants reliability review, but the evidence does not establish a specific failu

# What `verbose=True` shows

You will see:

### SQL Agent
- model name
- generated SELECT
- database rows

### RAG
- embedding model
- source PDF
- page
- similarity score
- retrieved text preview

### Final synthesis
- GPT-OSS 120B combines both sources

This is a useful **agent trace** for teaching. It does not display hidden chain-of-thought.

---

# Troubleshooting

### No Pydantic problem
This notebook does not import Pydantic or LlamaIndex.

### Qwen endpoint
Check:

`databricks-qwen3-embedding-0-6b`

### GPT-OSS endpoint
Check:

`databricks-gpt-oss-120b`

### A-001 not found
Run:

```sql
SELECT DISTINCT asset_id
FROM workspace.nuclear_enterprise_360.asset_360
LIMIT 50;
```

### PDF path
Check:

`/Workspace/Nuclear_Enterprise_360/A001 Documents`

In [0]:
# CELL 13 — Simple Gradio UI for testing the A001 SQL + RAG agent

%pip install -q gradio

import gradio as gr


def gradio_ask(question, history):
    """Run ask() and stream a simple trace back to the UI with error handling."""
    import io, traceback
    from contextlib import redirect_stdout

    if not question or not question.strip():
        history = history + [
            {"role": "user", "content": "(empty question)"},
            {"role": "assistant", "content": "⚠️ Please enter a question first."},
        ]
        return history, ""

    buf = io.StringIO()

    try:
        with redirect_stdout(buf):
            answer = ask(question, verbose=True)

        trace = buf.getvalue()
        result = f"""### Agent Trace
```
{trace}
```

---

### Final Answer

{answer}"""
    except Exception as e:
        trace = buf.getvalue()
        err_details = traceback.format_exc()
        result = f"""### Agent Trace
```
{trace}
```

---

### ⚠️ Error

```
{type(e).__name__}: {e}

{err_details}
``"""
        print(f"[GRADIO DEBUG] gradio_ask error: {type(e).__name__}: {e}")
        traceback.print_exc()

    history = history + [
        {"role": "user", "content": question},
        {"role": "assistant", "content": result},
    ]
    return history, ""


with gr.Blocks(title="A001 SQL + RAG Agent") as demo:
    gr.Markdown("# A001 SQL + RAG Agent")
    gr.Markdown("Ask a question about asset **A-001**. The agent runs SQL + RAG and returns a grounded answer.")

    chatbot = gr.Chatbot(height=500)
    question_box = gr.Textbox(
        label="Question",
        placeholder="e.g. What is the status of A-001 and what do the approved documents say about its inspection?",
        lines=2,
    )

    with gr.Row():
        send_btn = gr.Button("Ask", variant="primary")
        clear_btn = gr.Button("Clear")

    send_btn.click(gradio_ask, inputs=[question_box, chatbot], outputs=[chatbot, question_box])
    question_box.submit(gradio_ask, inputs=[question_box, chatbot], outputs=[chatbot, question_box])
    clear_btn.click(lambda: ([], ""), outputs=[chatbot, question_box])


demo.launch(server_name="0.0.0.0", share=True)

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
* Running on local URL:  http://0.0.0.0:7864
* Running on public URL: https://d8c1ed043561e45977.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
